### 拟牛顿法 Quasi-Newton Method 

#### 割线条件

拟牛顿法不需要显式计算Hessian, 而是利用相邻点的变量和梯度变化近似曲率。

定义：

$$ s_k=x_{k+1}-x_k, \quad y_k =\nabla f(x_{k+1})-\nabla f(x_k) \tag{1}$$


将梯度函数 $x_{k+1} $ 在 处一阶展开：

$$ \nabla f(x_k) \approx \nabla f(x_{k+1}) + \nabla ^2 f(x_{k+1}) (x_k-x_{k+1}) \tag{2} $$

移项整理得到：

$$ y_k \approx \nabla ^2 f(x_{k+1})s_k \tag{3}$$

构造 $B_{k+1}$ 近似 Hessian, 要满足割线条件

$$ B_{k+1}s_k=y_k \tag{4}$$

在(4)式基础上，令$G_{k+1}$ 近似 Hessian, 要求

$$ G_{k+1} y_k =s_k \tag{5} $$

一个向量等式不能唯一确定整个矩阵。因此有DFP更新，BFGS更新等方式。一般BFGS计算更稳健，应用也更广（补充 DFP更新 G 的形式与BFGS更新B的形式上一致，参数 G--B）。

#### BFGS更新 

先给出 更新$B$ 矩阵公式

$$ B_{k+1}=B_k+\frac{y_k y_k^T}{y_k^T s_k}-\frac{B_ks_k s_k^T B_k}{s_k ^TB_ks_k} \tag{6}$$


补充：
不妨 $B_{k+1}$ 是由 上一步矩阵 $B_k$  +修正矩阵$E_k$ (不是单位矩阵)

$$ B_{k+1}=B_k+ E_k \tag{7} $$

代入上式(4)， $ (B_k+E_k)s_k=y_k $

$$ E_k s_k=y_k-B_ks_k \tag{8} $$

(7)式表示 从构造$B$矩阵，到构造$E$矩阵。 (8)式表明 进一步拆分为构造两个矩阵 $A$ 和 $C$.

需要满足 $$ As_k=y_k, \quad C s_k= B_kS_k \tag{9} $$


构造矩阵$A$,  考虑 $y_k y_k^T$ , 乘上 $s_k$ ，有：

$$ \frac{y_k y_k^T s_k}{y_k^T s_k}= y_k \tag{10}$$

(10)式中 $y_k^T s_k$ 是标量， 整理 $ A=\frac{y_k y_k^T}{y_k^T s_k} $


构造矩阵 $C$, 考虑向量  $B_k s_k$ 同构造A的做法一致：

向量外积 $ (B_ks_k)(B_k s_k)^T $ ,若 $B_k$ 对称，$ (B_k s_k)^T=s_k^T B_k$

调整向量外积 $B_k s_k s_k^T B_k $, 再乘上 $s_k$ , 处理数乘 ($s_k^T B_k s_k$ 是标量) ：

$$ \frac{B_k s_k s_k^T B_k s_k}{s_k^T B_k s_k}=B_k s_k \tag{11} $$

整理得 $ C=\frac{B_k s_k s_k^T B_k}{s_k^T B_k s_k} $

$A$ 和 $C$ 代入(7) 式，即(6)式。

$G$ 矩阵的更新

令 $\rho _k=\frac{1}{y_k^T s_k} $ 给出更新公式 ：

$$ G_{k+1}=(I-\rho _k s_k y_k^T) G_k (I-\rho _ky_k s_k^T) + \rho _k s_k s_k^T \tag{12} $$

验证是否满足割线条件(5)式， 乘上 $y_k$， 看第二个括号 ：

$$ (I-\rho _k y_k s_k ^T)y_k=y_k- \frac{y_k s_k^T y_k}{y_k^T s_k} \tag{13}$$

注意到 $s_k^T y_k= y_k ^T s_k$。 由此该项为0， 那么(12)式右端 第一项消去，变成 $G_{k+1}y_k=\frac{s_k s_k^T y_k}{y_k^T s_k}=s_k$

其构造思路： 令$V_k=I-\rho _k y_k s_k ^T$, 满足 $V_k y_k=0$ 

即旧的矩阵作用在$y_k$上消失，再加入新的作用$y_k$的信息，得到 $s_k$

#### 更新顺序

从$x_k$ 开始，计算 $g_k=\nabla f(x_k) $ 记 $ G_k \approx H_k ^{-1} $

计算搜索方向： $ p_k= - G_k g_k$,   初始 $G_0=I $, 即最初使用一次梯度下降

更新位置： $x_{k+1}=x_k+\alpha _k p_k $

更新 新位置的梯度： $g_{k+1}=\nabla f(x_{k+1})$

计算$s_k, y_k$ :   $s_k=x_{k+1}-x_k=\alpha_k p_k, \quad y_k=g_{k+1}-g_{k} $

更新 $G_{k+1}=(I-\rho _k s_k y_k^T) G_k (I-\rho _ky_k s_k^T) + \rho _k s_k s_k^T $

循环...


在实践中，不需要更新$B$，公式推导中由(4)过渡到(5)。

In [6]:
#import importlib
import model
#importlib.reload(model)

In [7]:
import numpy as np
from time import perf_counter

In [8]:
def bfgs(features,labels,max_iter=200,tolerance=1e-7):
    dimension=features.shape[1]
    weights=np.zeros(dimension)
    inverse_hessian=np.eye(dimension) #G_0=I
    start_time=perf_counter()

    for iter in range(1,max_iter+1):
        grad=model.gradient(weights,features,labels)
        if np.linalg.norm(grad)<=tolerance:
            break
        direction=-inverse_hessian @ grad #p 牛顿方向

        #检查下降方向 g^Tp<0 即下降，否则按梯度下降
        if np.dot(grad,direction) >=0.0:
            inverse_hessian=np.eye(dimension)
            direction=-grad
        alpha,local_step=armijo(weights,direction,features,labels)
        sk=alpha*direction #s_k=x_{k+1}-x_k
        next_weights=weights+sk
        next_grad=model.gradient(next_weights,features,labels)
        gradient_change=next_grad-grad #y_k=g_{k+1}-g_k

        #y_k . s_k  判断\rho _k的分母不能为0
        curvature=np.dot(gradient_change,sk)
        if curvature> 1e-10 * np.linalg.norm(gradient_change)*np.linalg.norm(sk):
            rho=1.0/curvature
            identity=np.eye(dimension)
            left=identity-rho*np.outer(sk,gradient_change) 
            inverse_hessian=(
                        left @ inverse_hessian @left.T 
                        +rho* np.outer(sk,sk)
            )
        else:
            inverse_hessian=np.eye(dimension)
        weights=next_weights
    return model.make_record("BFGS",weights,features,labels,iter,perf_counter()-start_time)


In [9]:
# 返回 当前步 要使用的 alpha, 以及 线搜索迭代数
def armijo(weights,direction,features,labels,initial_alpha=1.0,c1=1e-4,shrink=0.5):
    # 条件不成立  \alpha 收缩
    value=model.objective(weights,features,labels)
    grad=model.gradient(weights,features,labels)

    # \nabla f(x_k)^T p_k<0  p_k下降方向
    slope=np.dot(grad,direction)
    if slope>=0.0:
        raise ValueError("Armijo line search requires a descent direction")
    alpha=initial_alpha
    iters=0

    # f(x_{k+1})<=f(x_k)+c_1 \alpha_k \nabla f(x_k)^Tp_k
    while model.objective(weights+alpha*direction,features,labels)>value+c1*alpha*slope:
        alpha*=shrink
        iters+=1
        if alpha<1e-14:
            raise RuntimeError("Armijo line search failed: alpha is too small")
    return alpha, iters+1

In [ ]:
def test_bfgs():
    x_train, y_train, x_test, y_test, metadata = model.load_data()
    results=bfgs(x_train,y_train)
    accuracy, precision,recall, log_loss = model.classification_metrics(
            results["weights"], x_test, y_test)
    print(
        f"method: {results.get('method')}\n"
        f"accuracy: {accuracy:.4f}\n"
        f"precision: {precision:.4f}\n"
        f"recall: {recall:.4f}\n"
        f"log_loss: {log_loss:.4f}"
    )

test_bfgs()

method: BFGS
accuracy: 0.9547
precision: 0.7731
recall: 1.0000
log_loss: 0.4230
